In [ ]:
!python  --version

Python 3.11.13


In [ ]:
!python  --version

Python 3.11.13


In [ ]:
!pip install -q pycaret

In [ ]:
from pycaret.classification import *
from sklearn.metrics import f1_score
import pandas as pd

In [ ]:
import joblib

In [ ]:
df_train = pd.read_csv('/content/train (1).csv')
df_test = pd.read_csv('/content/test (1).csv')

In [ ]:
df_train.head()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,2017/12/23,14:55:37,25.81,26.88,25.50,26.25,12,13,62,45,0.09,0.08,0.09,0.08,535,0.438462,0,0,0
1,2017/12/25,09:31:36,25.06,25.06,24.56,25.38,7,8,38,25,0.07,0.04,0.06,0.11,355,0.000000,0,0,0
2,2017/12/26,07:27:23,25.06,25.06,24.56,25.25,1,1,10,6,0.08,0.05,0.06,0.09,350,0.000000,0,0,0
3,2017/12/22,17:59:35,26.31,26.50,25.94,26.38,148,235,178,10,1.73,0.09,0.17,0.07,930,2.588462,1,1,3
4,2018/01/10,19:23:13,25.63,25.63,25.31,25.69,0,0,0,0,0.07,0.05,0.05,0.09,545,-2.130769,0,0,0


In [ ]:
df_train.Room_Occupancy_Count.value_counts()

,count
Room_Occupancy_Count,
0,6582
2,599
3,555
1,367


In [ ]:
s = setup(data=df_train,target='Room_Occupancy_Count')

,Description,Value
0,Session id,8418
1,Target,Room_Occupancy_Count
2,Target type,Multiclass
3,Original data shape,"(8103, 19)"
4,Transformed data shape,"(8103, 25)"
5,Transformed train set shape,"(5672, 25)"
6,Transformed test set shape,"(2431, 25)"
7,Numeric features,16
8,Categorical features,2
9,Preprocess,True


In [ ]:
def marco_f1(y,y_pred,**kwags):
  return f1_score(y,y_pred,average='macro')
add_metric('macro_f1','Marco F1',marco_f1)
print("setting up custom metric completed")


setting up custom metric completed


In [ ]:
multi_mode = ['lr','knn','rf','ada']
best_models = compare_models(include=multi_mode,fold=10,verbose=False)

In [ ]:
type(best_models).__name__

'KNeighborsClassifier'

In [ ]:
metrics = pull()

In [ ]:
metrics.columns

Index(['Model', 'Accuracy', 'AUC', 'Recall', 'Prec.', 'F1', 'Kappa', 'MCC',
       'Marco F1', 'TT (Sec)'],
      dtype='object')

In [ ]:
metrics.loc[:,['Accuracy','Marco F1']]

,Accuracy,Marco F1
knn,0.9967,0.9877
lr,0.9919,0.9744
rf,0.9600,0.8708
ada,0.8122,0.2241


In [ ]:
custom_grid = {
    'n_neighbors':[3,5,7,9,11,13,15]
}

tuned_model = tune_model(best_models,custom_grid=custom_grid,fold=10,optimize='Marco F1',verbose=False)

In [ ]:
for k,res in best_models.get_params().items():
  tuned = tune_model.get_params(k)
  if res != tuned:
    print(f"{k} : {res} --> {tuned}")



AttributeError: 'function' object has no attribute 'get_params'

In [ ]:
tuned_model_metrics = pull()

In [ ]:
tuned_model_metrics.loc[['Mean'],['Accuracy','Marco F1']]

,Accuracy,Marco F1
Fold,,
Mean,0.9967,0.9877


In [ ]:
final_model = finalize_model(tuned_model)

In [ ]:
prediction = predict_model(final_model,data=df_test)

In [ ]:
prediction


,id,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,...,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,prediction_label,prediction_score
0,ROOM-00001,2017/12/23,14:08:08,26.129999,27.129999,25.500000,26.379999,160,247,75,...,0.56,0.90,0.30,0.24,640,1.415385,0,0,2,1.0
1,ROOM-00002,2017/12/23,07:25:17,25.059999,25.059999,24.440001,25.379999,1,1,10,...,0.08,0.06,0.06,0.06,355,0.000000,0,0,0,1.0
2,ROOM-00003,2018/01/11,02:46:25,25.190001,25.190001,24.690001,25.250000,0,0,0,...,0.07,0.05,0.05,0.08,345,0.000000,0,0,0,1.0
3,ROOM-00004,2017/12/24,03:01:40,25.309999,25.309999,24.750000,25.690001,0,0,0,...,0.07,0.05,0.06,0.07,360,-0.096154,0,0,0,1.0
4,ROOM-00005,2017/12/23,19:57:53,26.190001,26.000000,25.879999,26.309999,0,0,0,...,0.07,0.05,0.05,0.07,1010,0.896154,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021,ROOM-02022,2017/12/26,08:11:48,25.059999,25.059999,24.559999,25.190001,5,5,26,...,0.08,0.05,0.06,0.11,350,0.000000,0,0,0,1.0
2022,ROOM-02023,2018/01/11,02:54:04,25.190001,25.190001,24.750000,25.250000,0,0,0,...,0.06,0.05,0.05,0.08,345,0.000000,0,0,0,1.0
2023,ROOM-02024,2017/12/26,01:51:25,25.309999,25.379999,24.940001,25.750000,0,0,0,...,0.08,0.05,0.07,0.10,355,0.000000,0,0,0,1.0
2024,ROOM-02025,2017/12/26,06:35:49,25.190001,25.190001,24.629999,25.379999,0,0,0,...,0.07,0.04,0.07,0.10,355,-0.019231,0,0,0,1.0


In [ ]:
prediction[['prediction_label']]

,prediction_label
0,2
1,0
2,0
3,0
4,0
...,...
2021,0
2022,0
2023,0
2024,0


In [ ]:
model_bundle = {
    'model':final_model,
    'features':df_train.columns.tolist(),
    'target':'Room_Occupancy_Count',
    'classes': final_model.classes_.tolist()
}


In [ ]:
save = joblib.dump(model_bundle,'Room_Occupancy_prediction.joblib')